<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-11/Aula11_LLM_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PARTE 1 — LLM Puro no Colab**

## **1- Configuração do ambiente**

In [ ]:
# Célula 0 — Instalar dependências
#!pip install transformers torch sentencepiece sentence-transformers faiss-cpu PyPDF2 -q
!pip install "transformers==4.46.3" torch sentencepiece sentence-transformers faiss-cpu PyPDF2 -q

import warnings
warnings.filterwarnings('ignore')

print('✅ Ambiente pronto!')

## **2- Geração de texto livre com GPT-2 em português**

In [ ]:
from transformers import pipeline

# Carregar modelo gerador
gerador = pipeline(
    'text-generation',
    model       = 'pierreguillou/gpt2-small-portuguese',
    max_length  = 120,
    do_sample   = True,
    temperature = 0.8,
    top_p       = 0.9,
)

In [ ]:
prompts = [
    'A inteligência artificial está transformando o mercado de trabalho porque',
    'O processamento de linguagem natural permite que computadores',
    'Em 2030, os modelos de linguagem serão capazes de',
]

In [ ]:
print('=== Geração de Texto Livre (GPT-2 PT) ===\n')
for prompt in prompts:
    resultado = gerador(prompt, num_return_sequences=1)[0]['generated_text']
    print(f'Prompt: "{prompt}"')
    print(f'Gerado: "{resultado}"')
    print()

## **3- Respondendo perguntas com Flan-T5**

O Flan-T5 foi treinado com instrução explícita — responde melhor a perguntas diretas:

In [ ]:
# Flan-T5 — excelente para QA sem contexto externo
qa_llm = pipeline(
    'text2text-generation',
    model     = 'google/flan-t5-base',
    max_length = 200,
)

In [ ]:
perguntas_llm = [
    'What is natural language processing?',
    'When was BERT released?',
    'What is the capital of Brazil?',
    'Who created the Transformer architecture?',
]

In [ ]:
print('=== QA com Flan-T5 (sem base de documentos) ===\n')
for pergunta in perguntas_llm:
    resposta = qa_llm(pergunta)[0]['generated_text']
    print(f'P: {pergunta}')
    print(f'R: {resposta}')
    print()

## **4- Testando a alucinação do LLM puro**

In [ ]:
# Perguntas sobre fatos específicos que o modelo pode não conhecer
perguntas_alucinacao = [
    'Qual é o CNPJ da Petrobras?',
    'Quem é o atual CEO da Vale em outubro de 2026?',
    'Qual foi o resultado do Flamengo no último jogo?',
    'Quais são os dados financeiros do relatório Q3 2026 da Ambev?',
]

In [ ]:
print('=== Demonstração de Alucinação — LLM sem base de documentos ===\n')
print('⚠️  Atenção: as respostas abaixo podem ser inventadas pelo modelo!\n')

for pergunta in perguntas_alucinacao:
    # Usar Flan-T5 em inglês para demonstrar
    prompt_en = f'Answer this question: {pergunta}'
    resposta = qa_llm(prompt_en)[0]['generated_text']
    print(f'P: {pergunta}')
    print(f'R: {resposta}')
    print(f'⚠️  Verifique: essa resposta é confiável sem uma fonte?')
    print()

# **PARTE 2 - RAG com Corpus em Variável**

## **1- Arquitetura completa do RAG**



```
┌─────────────────────────────────────────────────────────────────┐
│                         INDEXAÇÃO (offline)                     │
│                                                                 │
│  Documentos → Chunks → Embeddings → Índice vetorial (FAISS)    │
└─────────────────────────────────────────────────────────────────┘
                              ↓ (feito uma vez)
┌─────────────────────────────────────────────────────────────────┐
│                         CONSULTA (online)                       │
│                                                                 │
│  Pergunta → Embedding → Busca no índice → Top-K chunks         │
│                ↓                                               │
│  Montar prompt: [SYSTEM] + [CONTEXTO] + [PERGUNTA]             │
│                ↓                                               │
│  LLM gera a resposta fundamentada nos chunks recuperados       │
└─────────────────────────────────────────────────────────────────┘

```


## **2- Componentes implementados**

**Retriever**

Usa `sentence-transformers` para gerar embeddings semânticos e **FAISS** para busca de similaridade eficiente:

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Modelo de embeddings multilíngue — funciona bem em português
embedder = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)
print(f'Modelo de embeddings carregado!')
print(f'Dimensão dos vetores: {embedder.get_sentence_embedding_dimension()}')


**Generator**

Usa `Flan-T5` como gerador de respostas:

In [ ]:
from transformers import pipeline

gerador_rag = pipeline(
    'text2text-generation',
    model     = 'google/flan-t5-base',
    max_length = 300,
)
print('Gerador RAG carregado!')


## **3- RAG com corpus em variável**

**O caso mais simples**: a base de conhecimento está diretamente no código.

In [ ]:
# ── Base de conhecimento ─────────────────────────────────────
corpus = [
    "A Petrobras foi fundada em 3 de outubro de 1953 pelo presidente "
    "Getúlio Vargas, com o objetivo de garantir o monopólio estatal "
    "do petróleo no Brasil.",

    "A sede da Petrobras fica no Rio de Janeiro, no bairro da Cidade Nova. "
    "A empresa opera em mais de 27 países.",

    "Em 2026, a Petrobras registrou lucro líquido de R$ 45,3 bilhões, "
    "uma queda de 12% em relação ao ano anterior, devido à redução "
    "nos preços internacionais do barril de petróleo.",

    "O quadro de funcionários da Petrobras tem aproximadamente 45.000 "
    "colaboradores diretos. A empresa é listada na Bolsa de Valores "
    "de São Paulo (B3) e na Bolsa de Nova York (NYSE).",

    "A Petrobras é a maior empresa do Brasil por valor de mercado e "
    "uma das maiores produtoras de petróleo em águas profundas do mundo. "
    "O pré-sal representa mais de 70% da produção total da empresa.",

    "O presidente da Petrobras é nomeado pelo Conselho de Administração, "
    "cujos membros são indicados pelo governo federal, que é o acionista "
    "controlador com 36,6% das ações ordinárias.",
]

print(f'Corpus carregado: {len(corpus)} documentos')
for i, doc in enumerate(corpus):
    print(f'  [{i}] {doc[:70]}...')

In [ ]:
# ── Indexação — calcular embeddings e criar índice FAISS ─────
print('Indexando corpus...')

# Gerar embeddings para todos os documentos
embs_corpus = embedder.encode(corpus, show_progress_bar=True)

In [ ]:
embs_corpus

In [ ]:
# Criar índice FAISS (busca por L2 — equivalente a cosseno após normalização)
dimensao = embs_corpus.shape[1]
indice   = faiss.IndexFlatL2(dimensao)
indice.add(embs_corpus.astype('float32'))

In [ ]:
print(f'\n✅ Índice criado!')
print(f'   Documentos indexados: {indice.ntotal}')
print(f'   Dimensão dos vetores: {dimensao}')

**Funções do pipeline RAG**

In [ ]:
def recuperar(pergunta, top_k=2):
    """Busca os top_k documentos mais relevantes para a pergunta."""
    emb_pergunta = embedder.encode([pergunta]).astype('float32')
    distancias, indices = indice.search(emb_pergunta, top_k)

    resultados = []
    for i in range(top_k):
        resultados.append({
            'texto':     corpus[indices[0][i]],
            'distancia': distancias[0][i],
            'indice':    indices[0][i]
        })
    return resultados


In [ ]:
def montar_prompt(pergunta, documentos_recuperados):
    """Monta o prompt com contexto para o LLM."""
    contexto = '\n\n'.join([
        f'[Documento {i+1}]: {doc["texto"]}'
        for i, doc in enumerate(documentos_recuperados)
    ])

    prompt = (
        f'Based on the context below, answer the question in Portuguese. '
        f'If the answer is not in the context, say "Não encontrei essa '
        f'informação nos documentos disponíveis."\n\n'
        f'Context:\n{contexto}\n\n'
        f'Question: {pergunta}\n\n'
        f'Answer:'
    )
    return prompt

In [ ]:
def rag(pergunta, top_k=2, verbose=False):
    """Pipeline RAG completo: recuperar → montar prompt → gerar."""
    # Passo 1: recuperar documentos relevantes
    docs = recuperar(pergunta, top_k=top_k)

    # Passo 2: montar o prompt
    prompt = montar_prompt(pergunta, docs)

    if verbose:
        print(f'Documentos recuperados:')
        for d in docs:
            print(f'  [idx={d["indice"]}, dist={d["distancia"]:.3f}] '
                  f'"{d["texto"][:60]}..."')
        print()

    # Passo 3: gerar resposta
    resposta = gerador_rag(prompt)[0]['generated_text']
    return resposta, docs

print('Funções RAG definidas!')

In [ ]:
# ── Testar o RAG ─────────────────────────────────────────────
perguntas_teste = [
    'Quando foi fundada a Petrobras?',
    'Qual foi o lucro da Petrobras em 2026?',
    'Quantos funcionários a empresa tem?',
    'Quem nomeia o presidente da Petrobras?',
    'O que é o pré-sal?',
]

print('=== RAG com Corpus em Variável ===\n')

for pergunta in perguntas_teste:
    resposta, docs = rag(pergunta, top_k=2, verbose=True)
    print(f'❓ Pergunta: {pergunta}')
    print(f'✅ Resposta: {resposta}')
    print('─' * 60)
    print()


In [ ]:
# ── Comparativo: LLM puro vs. RAG ────────────────────────────
pergunta_comp = 'Qual foi o lucro da Petrobras em 2026?'

# LLM puro — sem contexto
llm_puro = qa_llm(f'Answer in Portuguese: {pergunta_comp}')[0]['generated_text']

# RAG — com contexto
rag_resp, _ = rag(pergunta_comp, top_k=2)

print('=== Comparativo — mesma pergunta ===\n')
print(f'Pergunta: "{pergunta_comp}"\n')
print(f'❌ LLM puro (sem documentos):')
print(f'   {llm_puro}')
print()
print(f'✅ RAG (com documentos):')
print(f'   {rag_resp}')


# **PARTE 3 - RAG com Arquivo TXT**

## **1- Lendo e chunkeando um arquivo TXT**

In [ ]:
# ── Criar arquivo TXT de exemplo ─────────────────────────────
conteudo_txt = """
RELATÓRIO ANUAL — EMPRESA EXEMPLO S.A. — 2026

1. APRESENTAÇÃO
A Empresa Exemplo S.A. foi fundada em 15 de março de 1998 em São Paulo, SP.
Opera nos segmentos de tecnologia, consultoria e desenvolvimento de software.
Possui 1.200 colaboradores diretos e 300 parceiros estratégicos.

2. DESEMPENHO FINANCEIRO
No ano de 2026, a receita líquida totalizou R$ 850 milhões, representando
crescimento de 18% em relação ao exercício anterior (R$ 720 milhões em 2025).
O EBITDA alcançou R$ 210 milhões, com margem de 24,7%.
O lucro líquido foi de R$ 130 milhões, com margem líquida de 15,3%.

3. OPERAÇÕES
A empresa está presente em 12 estados brasileiros e 5 países.
Em 2026, foram assinados 47 novos contratos, com ticket médio de R$ 3,2 milhões.
O segmento de cloud computing representou 42% da receita total.
O segmento de consultoria representou 35% e software proprietário 23%.

4. RECURSOS HUMANOS
Foram contratados 180 novos colaboradores ao longo de 2026.
A taxa de turnover ficou em 8,2%, abaixo da média do setor (11,5%).
Investimos R$ 12 milhões em treinamento e desenvolvimento.
O Net Promoter Score (NPS) dos colaboradores foi de 72 pontos.

5. PERSPECTIVAS 2027
A empresa projeta crescimento de receita entre 20% e 25% para 2027.
Serão investidos R$ 80 milhões em P&D, com foco em inteligência artificial.
A expansão internacional prevê entrada nos mercados do Chile e Colômbia.
"""

# Salvar o arquivo
with open('/content/relatorio_2026.txt', 'w', encoding='utf-8') as f:
    f.write(conteudo_txt)

print('Arquivo TXT criado: /content/relatorio_2026.txt')
print(f'Tamanho: {len(conteudo_txt)} caracteres')


In [ ]:
def carregar_txt(caminho):
    """Lê um arquivo TXT e retorna o conteúdo."""
    with open(caminho, 'r', encoding='utf-8') as f:
        return f.read()

In [ ]:
def chunkear_por_paragrafo(texto, min_chars=50):
    """
    Divide o texto em chunks por parágrafo.
    Remove parágrafos muito curtos (títulos, linhas em branco).
    """
    paragrafos = texto.split('\n')
    chunks = []
    for p in paragrafos:
        p = p.strip()
        if len(p) >= min_chars:
            chunks.append(p)
    return chunks

In [ ]:
def chunkear_por_tamanho(texto, tamanho=300, sobreposicao=50):
    """
    Divide o texto em chunks de tamanho fixo com sobreposição.
    Sobreposição evita perder contexto nas bordas dos chunks.
    """
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fim = inicio + tamanho
        chunk = texto[inicio:fim]
        # Tentar cortar na última palavra completa
        if fim < len(texto):
            ultimo_espaco = chunk.rfind(' ')
            if ultimo_espaco > 0:
                chunk = chunk[:ultimo_espaco]
                fim   = inicio + ultimo_espaco
        chunks.append(chunk.strip())
        inicio = fim - sobreposicao
    return [c for c in chunks if len(c) > 30]

In [ ]:
# Carregar e chunkar o TXT
texto_txt = carregar_txt('/content/relatorio_2026.txt')
chunks_paragrafo = chunkear_por_paragrafo(texto_txt)
chunks_tamanho   = chunkear_por_tamanho(texto_txt, tamanho=250, sobreposicao=40)

print(f'Texto carregado: {len(texto_txt)} caracteres')
print(f'Chunks por parágrafo:  {len(chunks_paragrafo)}')
print(f'Chunks por tamanho:    {len(chunks_tamanho)}')
print()
print('Primeiros 3 chunks (por parágrafo):')
for i, c in enumerate(chunks_paragrafo[:3]):
    print(f'  [{i}] "{c[:80]}..."')

In [ ]:
# ── Indexar os chunks do TXT ─────────────────────────────────
print('Indexando chunks do TXT...')

embs_txt = embedder.encode(chunks_paragrafo, show_progress_bar=True)
indice_txt = faiss.IndexFlatL2(embs_txt.shape[1])
indice_txt.add(embs_txt.astype('float32'))

print(f'\n✅ Índice TXT criado: {indice_txt.ntotal} chunks')

In [ ]:
# ── RAG com TXT ──────────────────────────────────────────────
def rag_txt(pergunta, chunks, indice_v, top_k=2, verbose=False):
    """RAG sobre arquivo TXT."""
    emb_p = embedder.encode([pergunta]).astype('float32')
    dists, idxs = indice_v.search(emb_p, top_k)

    docs_rec = []
    for i in range(top_k):
        docs_rec.append({
            'texto':     chunks[idxs[0][i]],
            'distancia': dists[0][i]
        })

    if verbose:
        print('Chunks recuperados:')
        for d in docs_rec:
            print(f'  [dist={d["distancia"]:.3f}] "{d["texto"][:70]}..."')
        print()

    contexto = '\n\n'.join([
        f'[Trecho {i+1}]: {d["texto"]}'
        for i, d in enumerate(docs_rec)
    ])

    prompt = (
        f'Based on the context below, answer the question in Portuguese. '
        f'If the answer is not in the context, say: '
        f'"Não encontrei essa informação nos documentos."\n\n'
        f'Context:\n{contexto}\n\n'
        f'Question: {pergunta}\n\nAnswer:'
    )

    resposta = gerador_rag(prompt)[0]['generated_text']
    return resposta

In [ ]:
# Testar
perguntas_txt = [
    'Quando foi fundada a Empresa Exemplo S.A.?',
    'Qual foi a receita líquida em 2026?',
    'Quanto a empresa investiu em treinamento?',
    'Qual é a projeção de crescimento para 2027?',
    'Quantos colaboradores a empresa possui?',
]

print('=== RAG com Arquivo TXT ===\n')
for pergunta in perguntas_txt:
    resposta = rag_txt(pergunta, chunks_paragrafo, indice_txt, verbose=True)
    print(f'❓ {pergunta}')
    print(f'✅ {resposta}')
    print('─' * 60)
    print()


# **PARTE 4 - RAG com PDF**

## **1- Extraindo texto de um PDF**

In [ ]:
import PyPDF2
import io

# ── Criar um PDF de exemplo programaticamente ────────────────
# Para esta aula, simulamos o upload de um PDF
# No Colab, você pode usar: from google.colab import files; files.upload()

# Vamos criar um PDF simples com fpdf2
!pip install fpdf2 -q

In [ ]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'Manual de Produtos - TechStore 2026', align='C')
        self.ln(10)

pdf = PDF()
pdf.add_page()
pdf.set_font('Helvetica', size=11)

In [ ]:
conteudo_pdf = [
    ('1. NOTEBOOK ULTRAPRO X500', [
        'Processador: Intel Core i7 de 13a geracao, 4.2 GHz turbo.',
        'Memoria RAM: 16 GB DDR5 expansivel ate 64 GB.',
        'Armazenamento: SSD NVMe de 512 GB (opcao de 1 TB ou 2 TB).',
        'Tela: 15.6 polegadas IPS Full HD 144 Hz anti-reflexo.',
        'Bateria: 72 Wh, autonomia de ate 10 horas de uso moderado.',
        'Preco sugerido: R$ 4.299,00. Garantia: 12 meses.',
    ]),
    ('2. SMARTPHONE PIXEL PRO', [
        'Tela AMOLED de 6.7 polegadas, 120 Hz, resolucao 2K.',
        'Camera principal de 50 MP com estabilizacao optica.',
        'Bateria de 5.000 mAh com carregamento rapido de 67W.',
        'Processador octa-core 3.2 GHz com 12 GB de RAM.',
        'Armazenamento interno de 256 GB sem slot microSD.',
        'Preco sugerido: R$ 3.199,00. Garantia: 12 meses.',
    ]),
    ('3. SMART TV VISION 55', [
        'Tela QLED de 55 polegadas resolucao 4K Ultra HD.',
        'Sistema operacional Android TV 13 com Google Assistant.',
        'HDR10+ e Dolby Vision para qualidade de imagem superior.',
        'Sistema de som 2.1 com 40W RMS e Dolby Atmos.',
        'Conectividade: Wi-Fi 6, Bluetooth 5.2, 4 portas HDMI 2.1.',
        'Preco sugerido: R$ 2.799,00. Garantia: 12 meses.',
    ]),
    ('4. FONE BLUETOOTH AUDIMAX PRO', [
        'Driver de 40mm com resposta de frequencia 20 Hz a 20 kHz.',
        'Cancelamento ativo de ruido (ANC) com 3 niveis de intensidade.',
        'Autonomia de 30 horas com ANC ativado, 40 horas sem ANC.',
        'Carregamento rapido: 10 minutos = 3 horas de uso.',
        'Codec aptX HD para audio de alta resolucao via Bluetooth 5.3.',
        'Preco sugerido: R$ 899,00. Garantia: 6 meses.',
    ]),
]

In [ ]:
for titulo, itens in conteudo_pdf:
    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, titulo, ln=True)
    pdf.set_font('Helvetica', size=10)
    for item in itens:
        pdf.cell(0, 6, f'  - {item}', ln=True)
    pdf.ln(4)

pdf.output('/content/catalogo_produtos.pdf')
print('PDF criado: /content/catalogo_produtos.pdf')

In [ ]:
def extrair_texto_pdf(caminho_pdf):
    """Extrai texto de todas as páginas de um PDF."""
    texto_total = []

    with open(caminho_pdf, 'rb') as f:
        leitor = PyPDF2.PdfReader(f)
        n_paginas = len(leitor.pages)
        print(f'PDF carregado: {n_paginas} página(s)')

        for i in range(n_paginas):
            pagina = leitor.pages[i]
            texto  = pagina.extract_text()
            if texto and texto.strip():
                texto_total.append(texto)
                print(f'  Página {i+1}: {len(texto)} caracteres extraídos')

    return '\n'.join(texto_total)

In [ ]:
# Extrair texto do PDF
texto_pdf = extrair_texto_pdf('/content/catalogo_produtos.pdf')
print(f'\nTotal de texto extraído: {len(texto_pdf)} caracteres')
print('\nPrimeiros 300 caracteres:')
print(texto_pdf[:300])

In [ ]:
def chunkear_pdf(texto, tamanho=300, sobreposicao=50):
    """
    Divide o texto extraído do PDF em chunks com sobreposição.
    Versão corrigida — sem risco de loop infinito.
    """
    # Limpar quebras de linha excessivas típicas de PDFs
    texto_limpo = ' '.join(texto.split())
    total       = len(texto_limpo)

    chunks = []
    inicio = 0

    while inicio < total:
        fim   = min(inicio + tamanho, total)
        chunk = texto_limpo[inicio:fim]

        # Tentar cortar na última palavra completa
        # SOMENTE se ainda não chegou ao fim do texto
        if fim < total:
            ultimo_espaco = chunk.rfind(' ')
            if ultimo_espaco > 0:              # encontrou espaço
                chunk = chunk[:ultimo_espaco]

        chunk = chunk.strip()
        if len(chunk) > 30:
            chunks.append(chunk)

        # ✅ CORREÇÃO CRÍTICA: próximo início sempre avança
        # usa o tamanho REAL do chunk adicionado, não o fim calculado
        proximo = inicio + len(chunk) - sobreposicao
        if proximo <= inicio:          # garantia anti-loop
            proximo = inicio + tamanho
        inicio = proximo

    return chunks


# Testar
chunks_pdf = chunkear_pdf(texto_pdf, tamanho=250, sobreposicao=40)
print(f'Chunks gerados do PDF: {len(chunks_pdf)}')
print()
for i, chunk in enumerate(chunks_pdf):
    print(f'[{i}] "{chunk[:80]}..."')

In [ ]:
# ── Indexar e consultar o PDF ─────────────────────────────────
print('Indexando PDF...')
embs_pdf   = embedder.encode(chunks_pdf, show_progress_bar=True)
indice_pdf = faiss.IndexFlatL2(embs_pdf.shape[1])
indice_pdf.add(embs_pdf.astype('float32'))
print(f'\n✅ Índice PDF: {indice_pdf.ntotal} chunks\n')

In [ ]:
# Testar RAG com PDF
perguntas_pdf = [
    'Qual é o preço do notebook UltraPro X500?',
    'Quantas horas dura a bateria do fone Audimax Pro com ANC ativado?',
    'Qual é a resolução da Smart TV Vision 55?',
    'O smartphone Pixel Pro tem slot para cartão de memória?',
    'Qual produto tem garantia de 6 meses?',
]

In [ ]:
print('=== RAG com PDF — Catálogo de Produtos ===\n')

for pergunta in perguntas_pdf:
    resposta = rag_txt(pergunta, chunks_pdf, indice_pdf, top_k=2, verbose=True)
    print(f'❓ {pergunta}')
    print(f'✅ {resposta}')
    print('─' * 60)
    print()

In [ ]:
# ── Upload de PDF do usuário (opcional) ──────────────────────
# Descomente para fazer upload do seu próprio PDF no Colab

# from google.colab import files
# uploaded = files.upload()
# nome_arquivo = list(uploaded.keys())[0]
#
# texto_meu_pdf = extrair_texto_pdf(nome_arquivo)
# meus_chunks   = chunkear_pdf(texto_meu_pdf, tamanho=300, sobreposicao=50)
#
# embs_meu   = embedder.encode(meus_chunks, show_progress_bar=True)
# indice_meu = faiss.IndexFlatL2(embs_meu.shape[1])
# indice_meu.add(embs_meu.astype('float32'))
#
# minha_pergunta = input('Faça uma pergunta sobre o seu documento: ')
# resposta = rag_txt(minha_pergunta, meus_chunks, indice_meu, top_k=3, verbose=True)
# print(f'Resposta: {resposta}')


# **PARTE 5 - Comparativo LLM vs. RAG**

## **1- Teste sistemático nas três bases**

In [ ]:
# Testar a mesma pergunta nas três abordagens
cenarios = [
    {
        'pergunta': 'Qual foi o lucro em 2026?',
        'chunks':   chunks_paragrafo,
        'indice':   indice_txt,
        'fonte':    'Relatório TXT'
    },
    {
        'pergunta': 'Qual é o preço do notebook?',
        'chunks':   chunks_pdf,
        'indice':   indice_pdf,
        'fonte':    'Catálogo PDF'
    },
    {
        'pergunta': 'Quando foi fundada a empresa?',
        'chunks':   corpus,
        'indice':   indice,
        'fonte':    'Corpus variável'
    },
]

print('=== Comparativo LLM puro vs. RAG ===\n')

for cenario in cenarios:
    pergunta = cenario['pergunta']
    fonte    = cenario['fonte']

    # LLM puro
    llm_resp = qa_llm(f'Answer in Portuguese: {pergunta}')[0]['generated_text']

    # RAG
    rag_resp = rag_txt(
        pergunta,
        cenario['chunks'],
        cenario['indice'],
        top_k=2
    )

    print(f'📄 Fonte: {fonte}')
    print(f'❓ Pergunta: {pergunta}')
    print(f'🤖 LLM puro: {llm_resp}')
    print(f'📚 RAG:      {rag_resp}')
    print('─' * 65)
    print()


## **2- Tabela de decisão: quando usar cada abordagem**

In [ ]:
tabela = [
    ('Informação factual privada (relatórios, contratos)',
     '❌ Não tem acesso', '✅ Recupera dos docs'),
    ('Informação pública geral (história, ciência)',
     '✅ Conhece do treino', '✅ Ou confirma docs'),
    ('Dados numéricos precisos (preços, datas, %)',
     '⚠️ Pode alucinar', '✅ Extrai dos docs'),
    ('Raciocínio e síntese de múltiplos documentos',
     '✅ (com GPT-4/Claude)', '✅ Combina recuperação + geração'),
    ('Custo computacional',
     '⚡ Mais rápido (sem busca)', '🐢 Busca + geração'),
    ('Atualização de conhecimento',
     '❌ Limitado ao treino', '✅ Adicionar docs novos'),
    ('Transparência (citar fontes)',
     '❌ Não indica fontes', '✅ Indica qual chunk usou'),
]

print('=== Quando usar LLM puro vs. RAG ===\n')
print(f'{"Cenário":<50} {"LLM Puro":<22} {"RAG"}')
print('=' * 100)
for cenario, llm, rag_v in tabela:
    print(f'{cenario:<50} {llm:<22} {rag_v}')


# **PARTE 6 - Ética, Alucinação e Limitações**

## **1- Tipos de falha dos LLMs**

In [ ]:
# Demonstrar os principais tipos de falha
falhas = {
    'Alucinação factual': {
        'prompt': 'Who won the 2026 Brazilian presidential election?',
        'problema': 'Pode inventar um candidato e data com total confiança'
    },
    'Viés de dados de treino': {
        'prompt': 'Describe a typical software engineer.',
        'problema': 'Tende a descrever perfis sub-representados de forma enviesada'
    },
    'Sensibilidade ao prompt': {
        'prompt_a': 'Explain why Python is better than Java.',
        'prompt_b': 'Compare Python and Java objectively.',
        'problema': 'A mesma pergunta com framing diferente gera respostas opostas'
    },
}

print('=== Tipos de Falha dos LLMs ===\n')

for tipo, info in falhas.items():
    print(f'🔴 {tipo}')
    if 'prompt' in info:
        resp = qa_llm(info['prompt'])[0]['generated_text']
        print(f'   Prompt: "{info["prompt"]}"')
        print(f'   Resposta: "{resp}"')
    print(f'   ⚠️  Problema: {info["problema"]}')
    print()


## **2- Boas práticas para uso responsável**

✅ FAÇA:
   - Sempre citar as fontes dos documentos usados no RAG
   - Incluir instrução de "diga que não sabe" no prompt
   - Validar respostas numéricas/factuais com a fonte original
   - Usar temperatura baixa (0.1-0.3) para tarefas factuais
   - Limitar o contexto a documentos confiáveis e atualizados

❌ NÃO FAÇA:
   - Usar LLM puro para decisões críticas sem revisão humana
   - Confiar em datas, CPFs, CNPJs ou números gerados pelo modelo
   - Usar como fonte primária sem verificação
   - Assumir que RAG elimina completamente a alucinação
   - Ignorar vieses do modelo em aplicações que envolvam pessoas


## **3- Avaliando a qualidade do RAG**

In [ ]:
def avaliar_rag(pergunta, resposta_esperada, chunks, indice_v, top_k=2):
    """Avaliação simples: verifica se a resposta contém a informação esperada."""
    resposta_gerada, docs = rag_txt.__wrapped__(pergunta, chunks, indice_v, top_k)

    # Métrica simples: recall de tokens-chave
    tokens_esperados = set(resposta_esperada.lower().split())
    tokens_gerados   = set(resposta_gerada.lower().split())
    tokens_comuns    = tokens_esperados & tokens_gerados
    recall           = len(tokens_comuns) / len(tokens_esperados) if tokens_esperados else 0

    return {
        'pergunta':         pergunta,
        'esperada':         resposta_esperada,
        'gerada':           resposta_gerada,
        'recall_tokens':    recall,
        'docs_recuperados': len(docs)
    }

# Conjunto de avaliação
print('=== Avaliação de Qualidade do RAG ===\n')
print('Nota: avaliação de LLMs é um campo complexo.')
print('Métricas robustas incluem ROUGE, BLEU, RAGAS e avaliação humana.\n')

avaliacao = [
    ('Quando foi fundada a Petrobras?',
     '1953 Getúlio Vargas outubro'),
    ('Qual foi o lucro da Petrobras em 2026?',
     '45 bilhões queda 12%'),
]

for pergunta, esperada in avaliacao:
    resp, docs = rag(pergunta, top_k=2)
    tokens_esp = set(esperada.lower().split())
    tokens_ger = set(resp.lower().split())
    recall     = len(tokens_esp & tokens_ger) / len(tokens_esp)
    print(f'P: {pergunta}')
    print(f'Esperado (tokens): {esperada}')
    print(f'Gerado:            {resp}')
    print(f'Recall de tokens:  {recall:.2f}')
    print()
